
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #D32F2F; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Cleanup</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Reset or Remove Course Resources</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">Use this notebook to start over or clean up after completing the course.</div>
</div>

</div>


<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">
<div style="padding: 18px 24px; background: #FFF3E0; border: 3px solid #FF9800; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <div style="font-weight: 700; margin-bottom: 8px; font-size: 16pt;">Before you run anything</div>
    <p>This notebook has <strong>two options</strong>. Read both and choose the one that fits your situation.</p>
    <ul style="padding-left: 20px; margin: 8px 0;">
      <li><strong>Option A: Mid-Course Reset</strong> — Drops all tables but keeps the schema and CSV files so you can restart from Lesson 1. Use this if you feel stuck and want a clean slate.</li>
      <li><strong>Option B: Full Cleanup</strong> — Removes <em>everything</em>: all tables, the volume, the CSV files, and the schema itself. Use this when you're completely finished with the course.</li>
    </ul>
    <p><strong>Run only the option you need.</strong> Do not run both.</p>
  </div>
</div>
</div>

### Setup
Run this cell first — it connects to the course catalog and schema so the cleanup commands work correctly.

In [0]:
%run ./Includes/Classroom-Setup-1

In [0]:
print(f"Connected to {my_catalog}.{my_schema}")

---
## Option A: Mid-Course Reset
**Use this if you want to start over from Lesson 1.**

This will:
- Drop all tables created during the course (lessons and practices)
- Drop any temporary views
- Keep the schema, volume, and CSV files intact
- After running this, go back to Lesson 1 and run the Setup cell. You'll be right back at the beginning


<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">
<div style="padding: 14px 20px; background: #FFF3E0; border-left: 4px solid #FF9800; border-radius: 6px;">
  <div style="font-size: 14pt; color: #0b2026;">
    <strong>Running the next cell will drop all tables.</strong> Your CSV source files will be preserved.
  </div>
</div>
</div>

In [0]:
# -----------------------------------------------
# OPTION A: Drop all tables, keep schema and files
# -----------------------------------------------

tables_to_drop = [
    # Lesson 01
    "employees",
    # Practice 01L
    "new_employees",
    # Lesson 04
    "current_employees_ctas",
    "current_employees_ui",
    # Practice 04L
    "new_hires_ctas",
    # Lesson 05
    "current_employees_copyinto",
    # Practice 05L
    "practice_copyinto",
    # Lesson 06
    "current_employees_bronze",
    "current_employees_silver",
    "total_roles_gold",
    # Practice 06L
    "practice_bronze",
    "practice_silver",
    "country_count_gold",
    # Lesson 07 / Job tasks
    "current_employees_bronze_job",
    "current_employees_silver_job",
    "total_roles_gold_job",
    # Lesson 08 / SDP pipeline
    "current_employees_bronze_sdp",
    "current_employees_silver_sdp",
    "total_roles_gold_sdp",
]

print("Dropping tables...\n")
for table in tables_to_drop:
    try:
        spark.sql(f"DROP TABLE IF EXISTS `{my_catalog}`.`{my_schema}`.`{table}`")
        print(f"  Dropped: {table}")
    except Exception as e:
        print(f"  Skipped: {table} ({e})")

# Drop temp views (only active in current session, but just in case)
for view in ["temp_total_roles", "temp_country_counts"]:
    try:
        spark.sql(f"DROP VIEW IF EXISTS {view}")
    except:
        pass

print("\n--- Reset complete ---")
print(f"All tables removed from {my_catalog}.{my_schema}.")
print("The schema, volume, and CSV files are still intact.")
print("You can now go back to Lesson 1 and start fresh.")

**Verify:** Run the cell below to confirm no tables remain.

In [0]:
%sql
SHOW TABLES;

---
## Option B: Full Cleanup
**Use this when you are completely finished with the course.**

This will remove **everything** created during the course:
- All tables
- The `myfiles` volume and its CSV files
- The `get_started_de` schema itself

After running this, the course environment will be completely gone. To take the course again, you would need to start from scratch by running the Setup cell in Lesson 1.


<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">
<div style="padding: 18px 24px; background: #FFEBEE; border: 3px solid #D32F2F; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <div style="font-weight: 700; margin-bottom: 8px; font-size: 16pt;">This action is irreversible</div>
    <p>Running the next cell will permanently delete:</p>
    <ul style="padding-left: 20px; margin: 8px 0;">
      <li>All tables in <code>labuser.get_started_de</code></li>
      <li>The <code>myfiles</code> volume and its contents (<code>employees.csv</code>, <code>employees2.csv</code>)</li>
      <li>The <code>get_started_de</code> schema</li>
    </ul>
    <p>Only run this if you are done with the course and want to free up the resources.</p>
  </div>
</div>
</div>

In [0]:
# -----------------------------------------------
# OPTION B: Remove everything — schema and all contents
# -----------------------------------------------

print("Starting full cleanup...\n")

# Drop all tables first (CASCADE on schema should handle this,
# but being explicit ensures clean error messages)
tables_to_drop = [
    "employees", "new_employees",
    "current_employees_ctas", "current_employees_ui", "new_hires_ctas",
    "current_employees_copyinto", "practice_copyinto",
    "current_employees_bronze", "current_employees_silver",
    "practice_bronze", "practice_silver",
    "total_roles_gold", "country_count_gold",
    "current_employees_bronze_job", "current_employees_silver_job", "total_roles_gold_job",
    "current_employees_bronze_sdp", "current_employees_silver_sdp", "total_roles_gold_sdp",
]

print("Dropping tables...")
for table in tables_to_drop:
    try:
        spark.sql(f"DROP TABLE IF EXISTS `{my_catalog}`.`{my_schema}`.`{table}`")
    except:
        pass

# Drop the volume (removes CSV files)
print("Dropping volume and files...")
try:
    spark.sql(f"DROP VOLUME IF EXISTS `{my_catalog}`.`{my_schema}`.myfiles")
    print("  Volume 'myfiles' dropped (employees.csv and employees2.csv removed)")
except Exception as e:
    print(f"  Could not drop volume: {e}")

# Drop the schema
print("Dropping schema...")
try:
    spark.sql(f"DROP SCHEMA IF EXISTS `{my_catalog}`.`{my_schema}` CASCADE")
    print(f"  Schema '{my_schema}' dropped")
except Exception as e:
    print(f"  Could not drop schema: {e}")

print("\n--- Full cleanup complete ---")
print(f"The schema '{my_catalog}.{my_schema}' and all its contents have been removed.")
print("To retake the course, run the Setup cell in Lesson 1 to recreate everything.")

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>